In [ ]:
import duckdb

**1. Campaign Attribution**
Determine which campaigns drove the most wallet addresses to the platform. Document your attribution logic clearly — the assumptions you made, the decisions you faced, and why you made them the way you did.
You may present your results however you like, but one mandatory output is a campaign_id column added to pixel_data, populated with the attributed campaign for each event.


In [ ]:
# Note: Original production data was removed for confidentiality.
# Using sample mock data with an identical schema for demonstration purposes.

con = duckdb.connect()
con.execute("""
    CREATE OR REPLACE TABLE mapping AS SELECT * FROM read_csv_auto('mock_address_twitter_id_mapping.csv', all_varchar=true);
    CREATE OR REPLACE TABLE campaigns AS SELECT * FROM read_csv_auto('mock_campaigns_daily_activity.csv', all_varchar=true);
    CREATE OR REPLACE TABLE events AS SELECT * FROM read_csv_auto('mock_events.csv', all_varchar=true);
    CREATE OR REPLACE TABLE transaction AS SELECT * FROM read_csv_auto('mock_transactions.csv', all_varchar=true);
""")

In [ ]:
query = """
WITH extracted_data AS (
    SELECT
        regexp_extract(full_url, 'utm_campaign=([^&]+)', 1) AS campaign_name,
        regexp_extract(full_url, 'utm_source=([^&]+)', 1) AS source_name,
        regexp_extract(full_url, 'utm_medium=([^&]+)', 1) AS medium_name
    FROM events
    WHERE full_url LIKE '%utm_campaign=%'
      AND full_url LIKE '%utm_source=%'
      AND full_url LIKE '%utm_medium=%'
)
SELECT
    campaign_name,
    source_name,
    medium_name,
    COUNT(*) as total_clicks
FROM extracted_data
GROUP BY
    campaign_name,
    source_name,
    medium_name
ORDER BY total_clicks DESC;
"""

In [ ]:
display(con.execute(query).df())

**Assumptions:**
If a user ID appears multiple times with a link pointing to a specific campaign, it will only be counted once as a unique attribution which was the first.


In [ ]:
query = """
WITH unique_campaigns AS (
    SELECT
        campaign_id, utm_campaign, utm_source, utm_medium
    FROM campaigns
    GROUP BY campaign_id, utm_campaign, utm_source, utm_medium
),

extracted_utms AS (
    SELECT
        user_id,
        event_time,
        regexp_extract(full_url, 'utm_campaign=([^&]+)', 1) AS campaign_name,
        regexp_extract(full_url, 'utm_source=([^&]+)', 1) AS source_name,
        regexp_extract(full_url, 'utm_medium=([^&]+)', 1) AS medium_name
    FROM events
    WHERE full_url LIKE '%utm_campaign=%'
),

-- 3. Sequence user clicks chronologically
numbered_clicks AS (
    SELECT
        e.user_id,
        c.campaign_id,
        -- This window function assigns a sequential number to each user event, ordered by time
        ROW_NUMBER() OVER(PARTITION BY e.user_id ORDER BY e.event_time ASC) as click_number
    FROM extracted_utms e
    JOIN unique_campaigns c
        ON e.campaign_name = c.utm_campaign
        AND e.source_name = c.utm_source
        AND e.medium_name = c.utm_medium
),

-- 4. Extract only the first click
user_mapping AS (
    SELECT user_id, campaign_id
    FROM numbered_clicks
    WHERE click_number = 1
)

-- 5. Map the attributed campaign back to the user
SELECT
    ev.user_id,
    ev.event_time,
    ev.event_name,
    ev.full_url,
    map.campaign_id
FROM events ev
LEFT JOIN user_mapping map
    ON ev.user_id = map.user_id
ORDER BY ev.user_id, ev.event_time;
"""

display(con.execute(query).df().head(50))

In [ ]:
query = """
-- 1. Extract unique campaigns (as defined previously)
WITH unique_campaigns AS (
    SELECT
        campaign_id, utm_campaign, utm_source, utm_medium
    FROM campaigns
    GROUP BY campaign_id, utm_campaign, utm_source, utm_medium
),

-- 2. Extract UTM parameters
extracted_utms AS (
    SELECT
        user_id,
        event_time,
        regexp_extract(full_url, 'utm_campaign=([^&]+)', 1) AS campaign_name,
        regexp_extract(full_url, 'utm_source=([^&]+)', 1) AS source_name,
        regexp_extract(full_url, 'utm_medium=([^&]+)', 1) AS medium_name
    FROM events
    WHERE full_url LIKE '%utm_campaign=%'
),

-- 3. Sequence clicks
numbered_clicks AS (
    SELECT
        e.user_id,
        c.campaign_id,
        ROW_NUMBER() OVER(PARTITION BY e.user_id ORDER BY e.event_time ASC) as click_number
    FROM extracted_utms e
    JOIN unique_campaigns c
        ON e.campaign_name = c.utm_campaign
        AND e.source_name = c.utm_source
        AND e.medium_name = c.utm_medium
),

-- 4. Map user to their first click
user_mapping AS (
    SELECT user_id, campaign_id
    FROM numbered_clicks
    WHERE click_number = 1
)
-- 5. Count unique converted users per campaign
SELECT
    campaign_id,
    COUNT(*) AS user_count   -- one row per user here, so COUNT(*) = COUNT(DISTINCT user_id)
FROM user_mapping
GROUP BY campaign_id
ORDER BY user_count DESC;
"""

display(con.execute(query).df())

**2. On-Chain Attribution**
Knowing that a user visited the website and that we can link them to a wallet address does not mean they converted on-chain. Using a public blockchain API of your choice, check which of the attributed wallets actually interacted with the following contract on Ethereum:

`0xEXAMPLE9bD2B4ADddBc894D8697F5170800EAdeC`

Assess which campaigns drove not just website visits, but actual on-chain conversions.

# New Section

**Assumptions:**
Since only 3 campaigns successfully drove traffic, I evaluated their active date ranges.
The first ran from March to May, while the second and third ran from April to May.
Therefore, I will analyze all three during their overlapping active period, assuming it serves as a representative sample for identifying the campaign that drove the most blockchain activity.

In [ ]:
query = """
WITH CleanedEvents AS (
    SELECT DISTINCT
        user_id,
        split_part(regexp_extract(full_url, '[?&]twitterid=([^&]+)', 1), '.', 1) AS twitter_id
    FROM events
    WHERE full_url LIKE '%twitterid=%'
)
SELECT DISTINCT
    ce.user_id,
    ce.twitter_id,
    m.address
FROM CleanedEvents AS ce
JOIN mapping AS m
    ON ce.twitter_id = m.twitter_id
JOIN transaction AS t
    ON m.address = t.From;
"""

display(con.execute(query).df())

In [ ]:
query = """
-- 1. Identify Twitter ID
WITH CleanedEvents AS (
    SELECT DISTINCT
        user_id,
        split_part(regexp_extract(full_url, '[?&]twitterid=([^&]+)', 1), '.', 1) AS twitter_id
    FROM events
    WHERE full_url LIKE '%twitterid=%'
),

-- 2. Identify users who converted on-chain via the target smart contract
BlockchainUsers AS (
    SELECT DISTINCT
        ce.user_id,
        ce.twitter_id,
        m.address
    FROM CleanedEvents AS ce
    JOIN mapping AS m
        ON ce.twitter_id = m.twitter_id
    JOIN transaction AS t
        ON LOWER(TRIM(m.address)) = LOWER(TRIM(t.From))
        AND LOWER(TRIM(t.To)) = '0xexample9bd2b4adddbc894d8697f5170800eadec'
),

-- 3. Generate a distinct list of active campaigns
unique_campaigns AS (
    SELECT DISTINCT campaign_id, utm_campaign
    FROM campaigns
    WHERE utm_campaign IS NOT NULL
),

-- 4. Extract campaign names from event URLs
extracted_utms AS (
    SELECT
        user_id,
        CAST(event_time AS BIGINT) AS event_time,
        regexp_extract(full_url, 'utm_campaign=([^&]+)', 1) AS campaign_name
    FROM events
    WHERE full_url LIKE '%utm_campaign=%'
),

-- 5. Sequence clicks chronologically per user
numbered_clicks AS (
    SELECT
        e.user_id,
        c.campaign_id,
        ROW_NUMBER() OVER(PARTITION BY e.user_id ORDER BY e.event_time ASC) as click_number
    FROM extracted_utms e
    JOIN unique_campaigns c
        ON e.campaign_name = c.utm_campaign
),

-- 6. Isolate the first click (attribution)
user_mapping AS (
    SELECT DISTINCT user_id, campaign_id
    FROM numbered_clicks
    WHERE click_number = 1
),

-- 7. Join on-chain converters with their originating campaign
ConvertedUsersWithCampaign AS (
    SELECT DISTINCT
        b.user_id,
        b.twitter_id,
        b.address,
        um.campaign_id
    FROM BlockchainUsers AS b
    JOIN user_mapping AS um
        ON b.user_id = um.user_id
)

-- 8. Aggregate and count converted users per campaign
SELECT
    campaign_id,
    COUNT(DISTINCT user_id) AS converted_users_count
FROM ConvertedUsersWithCampaign
GROUP BY campaign_id
ORDER BY converted_users_count DESC;
"""

display(con.execute(query).df())

**3. ENS Resolution**
For all wallet addresses attributed to campaigns, resolve their ENS (Ethereum Name Service) names using the Alchemy API.


In [ ]:
query = """
-- 1. Identify Twitter ID
WITH CleanedEvents AS (
    SELECT DISTINCT
        user_id,
        split_part(regexp_extract(full_url, '[?&]twitterid=([^&]+)', 1), '.', 1) AS twitter_id
    FROM events
    WHERE full_url LIKE '%twitterid=%'
),

-- 2. Identify users who converted on-chain via the smart contract
BlockchainUsers AS (
    SELECT DISTINCT
        ce.user_id,
        ce.twitter_id,
        m.address
    FROM CleanedEvents AS ce
    JOIN mapping AS m
        ON ce.twitter_id = m.twitter_id
    JOIN transaction AS t
        ON LOWER(TRIM(m.address)) = LOWER(TRIM(t.From))
        AND LOWER(TRIM(t.To)) = '0xexample9bd2b4adddbc894d8697f5170800eadec'
),

-- 3. Generate a unique list of active campaigns
unique_campaigns AS (
    SELECT DISTINCT campaign_id, utm_campaign
    FROM campaigns
    WHERE utm_campaign IS NOT NULL
),

-- 4. Extract campaign names from event URLs
extracted_utms AS (
    SELECT
        user_id,
        CAST(event_time AS BIGINT) AS event_time,
        regexp_extract(full_url, 'utm_campaign=([^&]+)', 1) AS campaign_name
    FROM events
    WHERE full_url LIKE '%utm_campaign=%'
),

-- 5. Sequence clicks chronologically per user
numbered_clicks AS (
    SELECT
        e.user_id,
        c.campaign_id,
        ROW_NUMBER() OVER(PARTITION BY e.user_id ORDER BY e.event_time ASC) as click_number
    FROM extracted_utms e
    JOIN unique_campaigns c
        ON e.campaign_name = c.utm_campaign
),

-- 6. Isolate the first click for each user
user_mapping AS (
    SELECT DISTINCT user_id, campaign_id
    FROM numbered_clicks
    WHERE click_number = 1
)

-- 7. Extract user details, wallet, and associated campaign for ENS resolution
SELECT DISTINCT
    b.user_id,
    b.twitter_id,
    b.address,
    um.campaign_id
FROM BlockchainUsers AS b
JOIN user_mapping AS um
    ON b.user_id = um.user_id;
"""

display(con.execute(query).df());

In [ ]:
from web3 import Web3

# 1. Connect to Alchemy
alchemy_url = "https://eth-mainnet.g.alchemy.com/v2/your_api_key"
w3 = Web3(Web3.HTTPProvider(alchemy_url))

# Helper function for ENS Reverse Resolution of a wallet address
def get_ens_name(wallet_address):
    try:
        if wallet_address:
            # Ensure the address is strictly checksummed before querying Web3
            checksum_address = Web3.to_checksum_address(wallet_address)
            ens_name = w3.ens.name(checksum_address)
            return ens_name if ens_name else "No ENS"
        return "Invalid Address"
    except Exception as e:
        return "Error / Not Found"

# 2. SQL Query to fetch all wallet addresses attributed to campaigns
query = """
-- 1. Identify unique campaigns
WITH unique_campaigns AS (
    SELECT DISTINCT campaign_id, utm_campaign, utm_source, utm_medium
    FROM campaigns
    WHERE utm_campaign IS NOT NULL
),

-- 2. Extract the three UTM parameters from the URL
extracted_utms AS (
    SELECT
        user_id,
        CAST(event_time AS BIGINT) AS event_time,
        regexp_extract(full_url, 'utm_campaign=([^&]+)', 1) AS campaign_name,
        regexp_extract(full_url, 'utm_source=([^&]+)', 1) AS source_name,
        regexp_extract(full_url, 'utm_medium=([^&]+)', 1) AS medium_name
    FROM events
    WHERE full_url LIKE '%utm_campaign=%'
),

-- 3. Join using the full UTM triplet to prevent duplicates for identical names
numbered_clicks AS (
    SELECT
        e.user_id,
        c.campaign_id,
        ROW_NUMBER() OVER(PARTITION BY e.user_id ORDER BY e.event_time ASC) as click_number
    FROM extracted_utms e
    JOIN unique_campaigns c
        ON e.campaign_name = c.utm_campaign
        AND e.source_name = c.utm_source
        AND e.medium_name = c.utm_medium
),

-- 4. Map user to the originating campaign
user_mapping AS (
    SELECT user_id, campaign_id
    FROM numbered_clicks
    WHERE click_number = 1
),

-- 5. Identify Twitter IDs
CleanedEvents AS (
    SELECT DISTINCT
        user_id,
        split_part(regexp_extract(full_url, '[?&]twitterid=([^&]+)', 1), '.', 1) AS twitter_id
    FROM events
    WHERE full_url LIKE '%twitterid=%'
)

-- 6. Retrieve Final Attributes
SELECT DISTINCT
    ce.user_id,
    ce.twitter_id,
    m.address,
    um.campaign_id
FROM CleanedEvents AS ce
JOIN mapping AS m
    ON ce.twitter_id = m.twitter_id
JOIN user_mapping um
    ON ce.user_id = um.user_id
WHERE m.address IS NOT NULL;
"""

# 3. Fetch data into Python
raw_results = con.execute(query).fetchall()
# Row structure: (user_id, twitter_id, address, campaign_id)

# 4. Build a set to avoid duplicate addresses and minimize API calls
unique_addresses = set(row[2] for row in raw_results if row[2] is not None)
print(f"Resolving ENS names for {len(unique_addresses)} unique attributed addresses via Web3...")

# 5. Iterate and resolve ENS names via the blockchain
ens_mapping = {}
for address in unique_addresses:
    ens_mapping[address] = get_ens_name(address)

# 6. Create a new list appending the ENS name to the end of each row
final_results = []
for row in raw_results:
    address = row[2]
    ens_name = ens_mapping.get(address, "Unknown")
    final_results.append(row + (ens_name,))

print(f"\nFinished resolving ENS via Web3 for {len(final_results)} user records.")
print("\nSample Results (user_id, twitter_id, address, campaign_id, ens_name):")
for row in final_results:
    print(row)

**4. Visualization**
Using all the data you have collected and produced — pixel data, campaign activity, on-chain transactions, and ENS names — create 4 charts or tables that you find most interesting or insightful. There is no prescribed format; choose what you think best communicates the story in the data.


In [ ]:
import matplotlib.pyplot as plt

# ============================================================
# TASK 4 - VISUALIZATION & INSIGHTS
# ============================================================

# 1. Query campaign traffic data via SQL
query_traffic = """
WITH unique_campaigns AS (
    SELECT DISTINCT
        campaign_id,
        utm_campaign,
        utm_source,
        utm_medium
    FROM campaigns
    WHERE utm_campaign IS NOT NULL
),

extracted_utms AS (
    SELECT
        user_id,
        CAST(event_time AS BIGINT) AS event_time,
        regexp_extract(full_url, 'utm_campaign=([^&]+)', 1) AS campaign_name,
        regexp_extract(full_url, 'utm_source=([^&]+)', 1) AS source_name,
        regexp_extract(full_url, 'utm_medium=([^&]+)', 1) AS medium_name
    FROM events
    WHERE full_url LIKE '%utm_campaign=%'
      AND full_url LIKE '%utm_source=%'
      AND full_url LIKE '%utm_medium=%'
),

numbered_clicks AS (
    SELECT
        e.user_id,
        c.campaign_id,
        ROW_NUMBER() OVER (
            PARTITION BY e.user_id
            ORDER BY e.event_time ASC
        ) AS click_number
    FROM extracted_utms e
    JOIN unique_campaigns c
        ON e.campaign_name = c.utm_campaign
        AND e.source_name = c.utm_source
        AND e.medium_name = c.utm_medium
)

SELECT
    campaign_id,
    user_id
FROM numbered_clicks
WHERE click_number = 1;
"""

raw_traffic = con.execute(query_traffic).fetchall()


# 2. Query list of users who completed an on-chain conversion
query_conversions = """
WITH CleanedEvents AS (
    SELECT DISTINCT
        user_id,
        split_part(
            regexp_extract(full_url, '[?&]twitterid=([^&]+)', 1),
            '.',
            1
        ) AS twitter_id
    FROM events
    WHERE full_url LIKE '%twitterid=%'
)

SELECT DISTINCT
    ce.user_id
FROM CleanedEvents ce
JOIN mapping m
    ON ce.twitter_id = m.twitter_id
JOIN transaction t
    ON LOWER(TRIM(m.address)) = LOWER(TRIM(t.From))
WHERE LOWER(TRIM(t.To)) =
      '0xexample9bd2b4adddbc894d8697f5170800eadec';
"""

converted_users = set(
    row[0]
    for row in con.execute(query_conversions).fetchall()
)


# 3. Query wallet addresses that executed a transaction on the target smart contract
query_converted_addresses = """
SELECT DISTINCT
    LOWER(TRIM("From")) AS address
FROM transaction
WHERE LOWER(TRIM("To")) =
      '0xexample9bd2b4adddbc894d8697f5170800eadec';
"""

converted_addresses = set(
    row[0]
    for row in con.execute(query_converted_addresses).fetchall()
)


# 4. Map campaign IDs to clean, readable names for aesthetics
campaign_names = {
    '4f731889-3bce-5afe-b562-2e43b0fc711e': 'Campaign A',
    'dd652e52-6a2a-5699-96ad-fa4e8f03c515': 'Campaign B',
    '79a4f545-de5b-5d57-8f48-285a14e6b287': 'Campaign C'
}


# 5. Initialize data structures and dictionaries for aggregating statistics
traffic_dict = {cid: 0 for cid in campaign_names}
conv_dict = {cid: 0 for cid in campaign_names}

ens_conv_dict = {cid: {'Has ENS': 0, 'No ENS': 0} for cid in campaign_names}
ens_blockchain_dict = {cid: {'Has ENS': 0, 'No ENS': 0} for cid in campaign_names}

# 6. Define invalid or null ENS values
invalid_ens_values = {None, '', 'No ENS', 'Unknown', 'Error / Not Found', 'Invalid Address'}

# 7. Analyze ENS status for converted users
converted_user_has_ens = {}

for row in final_results:
    uid, twitter_id, address, cid, ens = row[0], row[1], row[2], row[3], row[4]

    if uid in converted_users and address is not None and address.lower().strip() in converted_addresses:
        has_ens = ens not in invalid_ens_values
        if uid not in converted_user_has_ens:
            converted_user_has_ens[uid] = has_ens
        elif has_ens:
            converted_user_has_ens[uid] = True

# 8. Analyze ENS status for all blockchain-linked users
blockchain_user_info = {}

for row in final_results:
    uid, twitter_id, address, cid, ens = row[0], row[1], row[2], row[3], row[4]

    if cid in campaign_names and address is not None:
        has_ens = ens not in invalid_ens_values
        if uid not in blockchain_user_info:
            blockchain_user_info[uid] = {'campaign_id': cid, 'has_ens': has_ens}
        else:
            if has_ens:
                blockchain_user_info[uid]['has_ens'] = True

# 9. Count unique traffic users per campaign
seen_traffic_users = set()
for row in raw_traffic:
    cid, uid = row[0], row[1]
    if cid in campaign_names and uid not in seen_traffic_users:
        seen_traffic_users.add(uid)
        traffic_dict[cid] += 1

# 10. Count converted users and update their ENS metrics
seen_conversion_users = set()
for row in raw_traffic:
    cid, uid = row[0], row[1]
    if cid in campaign_names and uid in converted_users and uid not in seen_conversion_users:
        seen_conversion_users.add(uid)
        conv_dict[cid] += 1
        if converted_user_has_ens.get(uid, False):
            ens_conv_dict[cid]['Has ENS'] += 1
        else:
            ens_conv_dict[cid]['No ENS'] += 1

# 11. Aggregate ENS data for all blockchain users
for uid, info in blockchain_user_info.items():
    cid = info['campaign_id']
    if info['has_ens']:
        ens_blockchain_dict[cid]['Has ENS'] += 1
    else:
        ens_blockchain_dict[cid]['No ENS'] += 1

# 12. Prepare data lists for charting
labels = list(campaign_names.values())
t_vals = [traffic_dict[cid] for cid in campaign_names]
c_vals = [conv_dict[cid] for cid in campaign_names]

cr_vals = [ (c / t * 100) if t > 0 else 0 for c, t in zip(c_vals, t_vals) ]

ens_conv_has = [ens_conv_dict[cid]['Has ENS'] for cid in campaign_names]
ens_conv_no = [ens_conv_dict[cid]['No ENS'] for cid in campaign_names]

ens_blockchain_has = [ens_blockchain_dict[cid]['Has ENS'] for cid in campaign_names]
ens_blockchain_no = [ens_blockchain_dict[cid]['No ENS'] for cid in campaign_names]


# 13. Plot the 4 summary charts
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Advertising Attribution Analysis - Key Insights', fontsize=20, fontweight='bold', y=0.98)

x = range(len(labels))
width = 0.35

# Set dynamic padding to avoid text overlapping the top
y_pad_t = max(t_vals) * 0.04 if t_vals else 1
y_pad_cr = max(cr_vals) * 0.04 if cr_vals else 1
y_pad_c = max(c_vals) * 0.04 if c_vals else 1

max_total_bc = max([ens_blockchain_has[i] + ens_blockchain_no[i] for i in range(len(labels))]) if labels else 1
y_pad_bc = max_total_bc * 0.04

# --- Chart 1: Traffic vs. Conversions ---
traffic_bars = axes[0, 0].bar([p - width / 2 for p in x], t_vals, width, label='Traffic (Users)', color='#3498db', edgecolor='black')
conversion_bars = axes[0, 0].bar([p + width / 2 for p in x], c_vals, width, label='Conversions', color='#2ecc71', edgecolor='black')
axes[0, 0].set_title('1. Traffic vs. Conversions', fontsize=14)
axes[0, 0].set_xticks(x)
axes[0, 0].set_xticklabels(labels, fontsize=12)
axes[0, 0].set_ylabel('Unique Users')
axes[0, 0].legend()

for bar in traffic_bars:
    height = bar.get_height()
    if height > 0:
        axes[0, 0].text(bar.get_x() + bar.get_width() / 2, height + y_pad_t, f'{int(height)}', ha='center', va='bottom', fontsize=10)

for bar in conversion_bars:
    height = bar.get_height()
    if height > 0:
        axes[0, 0].text(bar.get_x() + bar.get_width() / 2, height + y_pad_t, f'{int(height)}', ha='center', va='bottom', fontsize=10)

# --- Chart 2: Efficiency - Conversion Rate (%) ---
rate_bars = axes[0, 1].bar(x, cr_vals, color='#e67e22', edgecolor='black')
axes[0, 1].set_title('2. Efficiency: Conversion Rate (%)', fontsize=14)
axes[0, 1].set_xticks(x)
axes[0, 1].set_xticklabels(labels, fontsize=12)
axes[0, 1].set_ylabel('Percentage (%)')

for bar in rate_bars:
    value = bar.get_height()
    if value > 0:
        axes[0, 1].text(bar.get_x() + bar.get_width() / 2, value + y_pad_cr, f'{value:.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')

# --- Chart 3: Quality of Converted Users (ENS) ---
axes[1, 0].bar(x, ens_conv_has, label='Has ENS', color='#9b59b6', edgecolor='black')
axes[1, 0].bar(x, ens_conv_no, bottom=ens_conv_has, label='No ENS', color='#95a5a6', edgecolor='black')
axes[1, 0].set_title('3. Quality of Converted Users (ENS)', fontsize=14)
axes[1, 0].set_xticks(x)
axes[1, 0].set_xticklabels(labels, fontsize=12)
axes[1, 0].set_ylabel('Converted Users')
axes[1, 0].legend()

for i, p in enumerate(x):
    total_converted = ens_conv_has[i] + ens_conv_no[i]
    if total_converted > 0:
        ens_percentage = (ens_conv_has[i] / total_converted) * 100
        axes[1, 0].text(p, total_converted + y_pad_c, f'Total: {total_converted}\nENS: {ens_conv_has[i]} ({ens_percentage:.1f}%)', ha='center', va='bottom', fontsize=10, fontweight='bold')

# --- Chart 4: Quality of All Blockchain-Linked Users (ENS) ---
axes[1, 1].bar(x, ens_blockchain_has, label='Has ENS', color='#9b59b6', edgecolor='black')
axes[1, 1].bar(x, ens_blockchain_no, bottom=ens_blockchain_has, label='No ENS', color='#95a5a6', edgecolor='black')
axes[1, 1].set_title('4. Quality of All Blockchain-Linked Users (ENS)', fontsize=14)
axes[1, 1].set_xticks(x)
axes[1, 1].set_xticklabels(labels, fontsize=12)
axes[1, 1].set_ylabel('Blockchain-Linked Users')
axes[1, 1].legend()

for i, p in enumerate(x):
    total_blockchain_users = ens_blockchain_has[i] + ens_blockchain_no[i]
    if total_blockchain_users > 0:
        ens_percentage = (ens_blockchain_has[i] / total_blockchain_users) * 100
        axes[1, 1].text(p, total_blockchain_users + y_pad_bc, f'Total: {total_blockchain_users}\nENS: {ens_blockchain_has[i]} ({ens_percentage:.1f}%)', ha='center', va='bottom', fontsize=10, fontweight='bold')

# Adjust layout padding so subplots and titles don't clash
plt.tight_layout(pad=3.0)
plt.show()